# Setup

## Imports

In [ ]:
# matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# others
import airr
import colorspacious as cs
import numpy as np

# add repertoire_zoo
import repertoire_zoo.bird as br
import repertoire_zoo.giraffe as gr

## Setup directories - Read in data

In [ ]:
# Define project/job uuid
project_id = ... # project uuid
job_id = ... # jov uuid

# pull out repertoires and names
groups_of_interest = [ ... ] # insert groups to compare (to compare all replace `...` with `None`)

# identify control group(s) to compare against
control_group_names = [ ... ] # insert control groups (for no control replace `...` with `None`)

# Define data directories
data_dir = ... #/path/to/data/directory/
repcalc_dir = data_dir + 'vdjserver/'+job_id+'/'
analysis_dir = data_dir + 'analysis/'
fig_dir = analysis_dir + 'figures/'

# Read in repertoire and repertoire group info
# The default name (repertoires.airr.json) is being used, but custom airr.json files can
# be used for defining repertoires and repertoire groups. 
repertoires_airr_json_filename = 'repertoires.airr.json'
sample_info_df = br.read_repertoire_info_airr(repcalc_dir + repertoires_airr_json_filename)
repertoire_group_df = br.read_repertoire_group_info_airr(repcalc_dir + repertoires_airr_json_filename)
data = airr.read_airr(repcalc_dir + repertoires_airr_json_filename)

if groups_of_interest == [None]:
    repertoire_groups_temp = { obj['repertoire_group_id'] : obj for obj in data['RepertoireGroup'] }
else:
    repertoire_groups_temp = { obj['repertoire_group_id'] : obj for obj in data['RepertoireGroup'] if obj['repertoire_group_name'] in groups_of_interest }
repertoires = { obj['repertoire_id'] : obj for obj in data['Repertoire']}

order_map = {name: idx for idx, name in enumerate(groups_of_interest)}
repertoire_groups = dict(sorted(
    repertoire_groups_temp.items(),
    key=lambda item: order_map.get(item[1]['repertoire_group_name'], len(groups_of_interest))
))

# make groups tuple
groups_tuple = [
    (
        repertoire_groups[rep_group_id]['repertoire_group_id'],
        repertoire_groups[rep_group_id]['repertoire_group_name'],
        [reps.get('repertoire_id') for reps in repertoire_groups[rep_group_id]['repertoires']]
    ) for rep_group_id in repertoire_groups
]

for i, rep_group_id in enumerate(repertoire_groups):
    print(rep_group_id, i, repertoire_groups[rep_group_id]['repertoire_group_name'])

condition_counts = {e[1] : e[2] for e in groups_tuple}

## Colormaps

In [ ]:
# set hue order (use condition names)
# By default the hue_order will be the same as the groups of interest. 
# If this is different, redfine `hue_order` as a list of groups from `groups_of_interest`
hue_order = groups_of_interest

cmap = mpl.colormaps['nipy_spectral'] # gist_rainbow
cmin = 0.0
cmax = 1.0

# V
v_genes = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='v_call',
    level='gene'
)['gene'].unique()
v_genes_cmap = {
    v_gene : cmap(np.linspace(cmin,cmax,len(v_genes))[i])
    for i, v_gene in enumerate(v_genes)
}

v_subgroups = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='v_call',
    level='subgroup'
)['gene'].unique()
v_subgroup_cmap = {
    v_subgroup : cmap(np.linspace(cmin,cmax,len(v_subgroups))[i])
    for i, v_subgroup in enumerate(v_subgroups)
}

# J
j_genes = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='j_call',
    level='gene'
)['gene'].unique()
j_genes_cmap = {
    j_gene : cmap(np.linspace(cmin,cmax,len(j_genes))[i])
    for i, j_gene in enumerate(j_genes)
}

j_subgroups = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='j_call',
    level='subgroup'
)['gene'].unique()
j_subgroup_cmap = {
    j_subgroup : cmap(np.linspace(cmin,cmax,len(j_subgroups))[i])
    for i, j_subgroup in enumerate(j_subgroups)
}

## cond_cmap
conds = br.load_and_prepare_data_junction_aa(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='.igblast.makedb.allele.clone.group.',
    call_type='junction_aa_length'
)['condition'].unique()

# Colorblind safe colors and colorblind analysis
# number of colors = number of base groups to compare
base_colors = ['#A577F9', '#699B59', '#D36135', '#086788', '#FFBC42']
cond_cmap = {
    cond : base_colors[i]
    for i, cond in enumerate(conds)
}

names = [cond[1] for cond in groups_tuple]

deuteranomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "deuteranomaly",
        "severity": 100
    },
    end = "sRGB1"
)

protanomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "protanomaly",
        "severity": 100
    },
    end = "sRGB1"
)

tritanomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "tritanomaly",
        "severity": 100
    },
    end = "sRGB1"
)

fig, axes = plt.subplots(4, 1, figsize=(len(list(cond_cmap.keys())), 4))
colorblind_simulated_cmaps = [deuteranomaly_simulated_cmap, protanomaly_simulated_cmap, tritanomaly_simulated_cmap]
gr.show_colormap(cond_cmap, axes[0], False)
for i, (ax, colorblind_simulated_cmap) in enumerate(zip(axes[1:], colorblind_simulated_cmaps)):
    colorblind_simulated_cmap = [np.clip(e, 0, 1) for e in colorblind_simulated_cmap]
    colorblind_simulated_cmap = {k:v for k,v in zip(cond_cmap.keys(), colorblind_simulated_cmap)}
    if i != 2: gr.show_colormap(colorblind_simulated_cmap, ax, False)
    else: gr.show_colormap(colorblind_simulated_cmap, ax, True, 2.5)

axes[0].set_ylabel('Normal', rotation=0, labelpad=20)
axes[1].set_ylabel('Deuteranomaly', rotation=0, labelpad=39)
axes[2].set_ylabel('Protanomaly', rotation=0, labelpad=32)
axes[3].set_ylabel('Tritanomaly', rotation=0, labelpad=32)
fig.suptitle('Colorblind Analysis', fontsize=24, y=1.02)
fig.show()

# CDR3

## Bar graph of CDR3 length

In [ ]:
df = br.load_and_prepare_data_junction_aa(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='.igblast.makedb.allele.clone.group.',
    call_type='junction_aa_length'
)
    
fig, ax = gr.plot_junction_aa_length(
    df=df,
    x_col='CDR3_LENGTH',
    y_col='CDR3_RELATIVE',
    controls=control_group_names,
    hue_col='condition',
    palette=cond_cmap,
    legend=True,
    title={'t':'CDR3 Length', 'y':1.0, 'fontsize':20},
    figsize=(12,6)
)

# add N for labels
gr.add_N_to_legend(ax, groups_tuple)

fig.show()

## AA Sharing

In [ ]:
# plot_type = 'between' if len(groups) == 2 else 'within'
plot_type = 'between'

sharing = 'cdr3_aa_sharing'
processing_stage = 'repertoire.shared.matrix'
# processing_stage = 'group.summary_comparison'

similarity_metric = 'a_in_b' #jaccard, simpson_coefficient, a_in_b, b_in_a
# similarity_metric = 'b_in_a' #jaccard, simpson_coefficient, a_in_b, b_in_a
percentage = True

# setup to compare against healthy control (diag_hi_groups[0])
for control_group_name in control_group_names:
    for i, group in enumerate([group for group in groups_tuple if group[1] in hue_order]):
        if group[1] == control_group_name or control_group_name is None:
            control = group
            break

    for i, group in enumerate([group for group in groups_tuple if group[1] in hue_order]):
        if group == control:
            continue
        matrix = br.load_shared_cdr3_aa_data(
            repcalc_dir,
            repertoire_groups, 
            sample_info_df, 
            groups=[control, group], 
            percentage=percentage,
            similarity_metric=similarity_metric,
            sharing=sharing,
            processing_stage=processing_stage,
            plot_type=plot_type
        )
        matrix = matrix.sort_index().sort_index(axis=1)
        fig, ax = gr.plot_cdr3_aa_sharing_heatmap(matrix, cmap = 'viridis', figsize = (36, 30))
        plt.title(f"Heatmap of {sharing} {plot_type} group")
        plt.tight_layout()


## Sequence count

In [ ]:
df = br.load_cdr3_sequences_each_repertoire(
    repcalc_dir=repcalc_dir,
    repertoire_id=data['Repertoire'][0]['repertoire_id'],
    sharing='cdr3_aa_sharing',
    processing_stage='repertoire'
)

fig, ax = gr.plot_cdr3_sequence_count(
    df=df,
    abundance=True,
    top_n=20,
    figsize=(16,8)
)

fig.show()

## Unique number of sequences

In [ ]:
sharing='cdr3_aa_sharing'
processing_stage='repertoire'

# setup to compare against healthy control (diag_hi_groups[0])
for control_group in control_group_names:
    if control_group == None:
        control = groups_tuple[0]
    else:
        for i, group in enumerate([group for group in groups_tuple if group[1] in hue_order]):
            if group[1] == control_group:
                control = group
            break

    for i, group in enumerate([group for group in groups_tuple if group[1] in hue_order]):
        if group == control:
            continue

        df = br.load_unique_cdr3_sequence_numbers(
            repcalc_dir,
            repertoire_groups,
            groups=[control, group],
            sample_info_df=sample_info_df,
            sharing=sharing,
            processing_stage=processing_stage
        )
        fig, axes = gr.plot_unique_number_of_cdr3(
            df,
            # palette=cond_cmap,
            # figsize=(12, 6)
        )
        gr.add_N_to_legend(axes[1], groups_tuple)
        condition_counts = {e[1] : len(e[2]) for e in groups_tuple}
        fig.suptitle(f"Unique number of CDR3 for {' vs '.join([control[1]+f' (N={condition_counts[control[1]]})', group[1]+f' (N={condition_counts[group[1]]})'])}")
        fig.tight_layout()

        fig.show()

        # figname = f'Figures/unique_number_of_cdr3_{'_'.join(group_names)}.png'
        # plt.savefig(figname, bbox_inches = 'tight', dpi = 300)